# Previsões do extrator no conjunto de teste

Roda o Qwen2.5-1.5B-Instruct sem ajuste e com o adaptador QLoRA nos 60 trechos de `data/gold/test.jsonl` e salva duas coisas:

- `reports/<data>_qwen-base_test.json`: as métricas do modelo sem ajuste, a coluna que faltava no model card;
- `reports/<data>_previsoes_test.jsonl`: a resposta de cada modelo para cada trecho, ao lado da resposta certa, que alimenta o explorador da demo.

Antes de rodar, escolha a GPU T4 em Ambiente de execução → Alterar o tipo de ambiente de execução. Leva cerca de 40 minutos: uns 20 para cada modelo. A geração é gulosa, então as métricas do QLoRA devem bater com as de `reports/2026-11-01_qlora_test.json`.

In [ ]:
!rm -rf /content/lacuna && git clone -q https://github.com/juan-barross/lacuna.git /content/lacuna
%cd /content/lacuna
!pip install -q -e . "accelerate==1.15.0"
!pip install -q "transformers==5.22.1" "peft==0.21.2" "bitsandbytes==0.50.2"

O adaptador ainda é privado, então o login pede um token de leitura do Hugging Face.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import json
import sys
from datetime import date
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Sem GPU: escolha a T4 em Ambiente de execução → Alterar o tipo de ambiente de execução"

from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

sys.path.insert(0, "src")

from dxp.evaluation import score
from dxp.files import read_jsonl
from dxp.local_model import LocalExtractor
from dxp.schema import Ato

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "juan1211/lacuna-extrator-qwen2.5-1.5b-lora"
HOJE = date.today()

ouro = read_jsonl(Path("data/gold/test.jsonl"))
tok = AutoTokenizer.from_pretrained(BASE)
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
modelo = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=quant, device_map={"": 0}, dtype=torch.float16
)
modelo.eval()
print(len(ouro), "trechos no teste,", torch.cuda.get_device_name())

## Sem ajuste

O relatório do modelo sem ajuste é salvo assim que a rodada termina, para não se perder se o Colab desconectar durante a do QLoRA.

In [ ]:
def resumo(campos):
    return {campo: round(m["f1"], 2) for campo, m in campos.items()}


extrator_base = LocalExtractor(tok, modelo)
campos_base = score(tqdm(ouro, desc="sem ajuste"), extrator_base)
print(resumo(campos_base))

Path("reports").mkdir(exist_ok=True)
relatorio = {"campos": campos_base, "uso": {**extrator_base.usage(BASE), "gpu": torch.cuda.get_device_name()}}
Path(f"reports/{HOJE}_qwen-base_test.json").write_text(
    json.dumps(relatorio, indent=2, ensure_ascii=False), encoding="utf-8"
)

## Com QLoRA

O adaptador entra sobre o mesmo modelo base em 4 bits. A última linha compara o F1 de cada campo com o relatório de 01/11.

In [ ]:
modelo = PeftModel.from_pretrained(modelo, ADAPTER)
modelo.eval()
extrator_qlora = LocalExtractor(tok, modelo)
campos_qlora = score(tqdm(ouro, desc="QLoRA"), extrator_qlora)

anterior = resumo(json.loads(Path("reports/2026-11-01_qlora_test.json").read_text(encoding="utf-8"))["campos"])
agora = resumo(campos_qlora)
print(agora)
print("diferença para 01/11:", {campo: round(agora[campo] - anterior[campo], 2) for campo in agora})

## Salvar as previsões

In [ ]:
def resposta(answer):
    return answer.ato.model_dump(mode="json") if answer.ato else None


linhas = [
    {
        "ato_id": row["ato_id"],
        "ouro": Ato.model_validate(row).model_dump(mode="json"),
        "base": resposta(base),
        "qlora": resposta(qlora),
        "segundos": {"base": round(base.seconds, 2), "qlora": round(qlora.seconds, 2)},
    }
    for row, base, qlora in zip(ouro, extrator_base.answers, extrator_qlora.answers, strict=True)
]
with open(f"reports/{HOJE}_previsoes_test.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(linha, ensure_ascii=False) + "\n" for linha in linhas)
print(len(linhas), "trechos salvos")

In [ ]:
from google.colab import files

!zip -qj previsoes_test.zip reports/{HOJE}_qwen-base_test.json reports/{HOJE}_previsoes_test.jsonl
files.download("previsoes_test.zip")